# DWD Gridded Climate Data to a Regional Time Series

This notebook converts monthly gridded climate data from the German Weather Service (DWD) into a regional time series for a selected study area.

The example uses monthly soil moisture grids and clips them to the municipality of Kerpen. The same workflow can be adapted to other DWD monthly gridded datasets, such as precipitation, by changing the settings below.

## Workflow

1. Automate DWD raster file downloads.
2. Decompress `.asc.gz` files.
3. Convert `.asc` grids to GeoTIFF.
4. Reproject and clip GeoTIFFs to a study-area shapefile.
5. Calculate a regional monthly mean.
6. Export the result as a CSV time series.
7. Alternatively: Export the results as NetCDF.


## Notebook environment

This notebook is designed to run in Google Colab or another Jupyter-style environment.

Commands that start with `!` are shell commands. They are executed by the notebook environment rather than by Python directly. In this workflow, shell commands are used to install required system tools, download data, and run GDAL command-line utilities.

Additional Python packages and system tools are installed throughout the workflow when they are needed.

In [ ]:
# Uncomment to install required Python packages
# !pip install -q numpy pandas rasterio geopandas xarray rioxarray netcdf4

In [69]:
from pathlib import Path
import os
import glob
import re
import subprocess

import rasterio
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray as rxr

### Adapting the settings for precipitation

For monthly precipitation, the main changes would be:

```python
DWD_DATASET_FOLDER = "precipitation"
DATASET_LABEL = "Monthly Precipitation"
VALUE_COLUMN = "precipitation_mm"
VALUE_UNIT = "mm/month"
```

The later cells use `find` to search recursively through the downloaded DWD folder. This is useful because some DWD datasets store files directly in one folder, while others use subfolders.


## User settings

Edit this section to adapt the notebook to another DWD dataset, region, time period, or output name.

The rest of the notebook should usually not need to be changed.


In [67]:
# ---------------------------------------------------------------------
# USER SETTINGS
# ---------------------------------------------------------------------
# Change these values to adapt the workflow.
# ---------------------------------------------------------------------

REGION_NAME = "Kerpen"
START_YEAR = 1991
END_YEAR = 2025

# DWD dataset folder.
DWD_DATASET_FOLDER = "soil_moist"

# Human-readable metadata for tables and plots.
DATASET_LABEL = "Monthly Soil Moisture"
VALUE_COLUMN = "soil_moisture_nfk_percent"
VALUE_UNIT = "% nFK"

# Coordinate reference system assigned during conversion from ASC to GeoTIFF.
# This should match the CRS of the DWD grid and the study-area shapefile.
SOURCE_CRS = "EPSG:31467"

# Main working directory.
# In Google Colab this is usually /content.
WORK_DIR = Path("/content")

# DWD source URL.
DWD_URL = (
    "https://opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/"
    f"{DWD_DATASET_FOLDER}/"
)

# Local folder created by wget after downloading from the DWD server.
DWD_DATA_DIR = (
    WORK_DIR
    / "opendata.dwd.de"
    / "climate_environment"
    / "CDC"
    / "grids_germany"
    / "monthly"
    / DWD_DATASET_FOLDER
)

# Study-area shapefile.
# Update this path if your shapefile is stored somewhere else.
SHAPEFILE = (
    WORK_DIR
    / "shapefiles"
    / "kerpen_adm_25832"
    / "kerpen_adm_25832.shp"
)

# Output folders and files.
CROPPED_DIR = WORK_DIR / "cropped"
OUTPUT_CSV = WORK_DIR / f"{VALUE_COLUMN}_{REGION_NAME}_{START_YEAR}_{END_YEAR}.csv"

print("Dataset URL:", DWD_URL)
print("Local DWD folder:", DWD_DATA_DIR)
print("Shapefile:", SHAPEFILE)
print("Output CSV:", OUTPUT_CSV)

Dataset URL: https://opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/soil_moist/
Local DWD folder: /content/opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/soil_moist
Shapefile: /content/shapefiles/kerpen_adm_25832/kerpen_adm_25832.shp
Output CSV: /content/soil_moisture_nfk_percent_Kerpen_1991_2025.csv


In [19]:
# Pass Python variables to the shell environment.
# This makes them usable inside later `!` shell commands in Colab.
os.environ["DWD_DATA_DIR"] = str(DWD_DATA_DIR)
os.environ["SOURCE_CRS"] = str(SOURCE_CRS)

print("DWD_DATA_DIR:", os.environ["DWD_DATA_DIR"])
print("SOURCE_CRS:", os.environ["SOURCE_CRS"])

DWD_DATA_DIR: /content/opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/soil_moist
SOURCE_CRS: EPSG:31467


## 1. Download the DWD files

This downloads the selected DWD dataset from the DWD Open Data server.

In [2]:
!wget -r -np -R "index.html*" "{DWD_URL}"

# which is basically the same as
# !wget -r -np -R "index.html*" https://opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/soil_moist/
# change DWD_URL in the User Settings to download a different data set

--2026-07-09 10:17:58--  https://opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/soil_moist/
Resolving opendata.dwd.de (opendata.dwd.de)... 141.38.2.164
Connecting to opendata.dwd.de (opendata.dwd.de)|141.38.2.164|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/soil_moist/index.html.tmp’

opendata.dwd.de/cli     [ <=>                ]  64.98K   375KB/s    in 0.2s    

2026-07-09 10:17:59 (375 KB/s) - ‘opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/soil_moist/index.html.tmp’ saved [66539]

Loading robots.txt; please ignore errors.
--2026-07-09 10:17:59--  https://opendata.dwd.de/robots.txt
Reusing existing connection to opendata.dwd.de:443.
HTTP request sent, awaiting response... 404 Not Found
2026-07-09 10:17:59 ERROR 404: Not Found.

Removing opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/soil_moist/index.html.tmp since

## 2. Check the downloaded files

This quick check counts how many files were downloaded into the local DWD folder.


In [3]:
file_count = 0

for root, dirs, files in os.walk(DWD_DATA_DIR):
    file_count += len(files)

print(f"Files downloaded: {file_count}")

Files downloaded: 431


## 3.1 Inspect downloaded files

Before decompressing the DWD files, we first check which files were downloaded. This is useful because some DWD datasets store files directly in one folder, while others use monthly subfolders.

## 3.2 Decompress the downloaded raster files

DWD provides the rasters as compressed `.asc.gz` files, as you can see from the list of filenames. This step decompresses them to `.asc`.

The command searches recursively so that it also works for DWD datasets that are stored in subfolders.


In [5]:
## Inspect downloaded files

!find "{DWD_DATA_DIR}" -type f

/content/opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/soil_moist/BESCHREIBUNG_gridsgermany_monthly_soil_moist_de.pdf
/content/opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/soil_moist/grids_germany_monthly_soil_moist_202505.asc.gz
/content/opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/soil_moist/grids_germany_monthly_soil_moist_201705.asc.gz
/content/opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/soil_moist/grids_germany_monthly_soil_moist_201901.asc.gz
/content/opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/soil_moist/grids_germany_monthly_soil_moist_202211.asc.gz
/content/opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/soil_moist/grids_germany_monthly_soil_moist_199107.asc.gz
/content/opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/soil_moist/grids_germany_monthly_soil_moist_202203.asc.gz
/content/opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/soil_moist/grids_germ

In [6]:
!find "{DWD_DATA_DIR}" -name "*.gz" -exec gunzip -f {{}} \;

## 4. Inspect the data folder

This is a simple sanity check to see what is now available in the downloaded DWD folder.


In [8]:
# we can inspect the first 10 files, notice how ".asc.gz" is now ".asc"
os.listdir(DWD_DATA_DIR)[:10]

['grids_germany_monthly_soil_moist_202508.asc',
 'grids_germany_monthly_soil_moist_200706.asc',
 'grids_germany_monthly_soil_moist_202410.asc',
 'grids_germany_monthly_soil_moist_202210.asc',
 'BESCHREIBUNG_gridsgermany_monthly_soil_moist_de.pdf',
 'grids_germany_monthly_soil_moist_199809.asc',
 'grids_germany_monthly_soil_moist_199806.asc',
 'grids_germany_monthly_soil_moist_199708.asc',
 'grids_germany_monthly_soil_moist_200509.asc',
 'grids_germany_monthly_soil_moist_200009.asc']

## 5. Install GDAL

GDAL is used for raster conversion and clipping.

In Google Colab, this installation step is usually needed. If you run the notebook locally and GDAL is already installed, you can skip this cell.


In [9]:
!apt-get install -y gdal-bin

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  python3-gdal python3-numpy
Suggested packages:
  libgdal-grass python-numpy-doc python3-dev python3-pytest
The following NEW packages will be installed:
  gdal-bin python3-gdal python3-numpy
0 upgraded, 3 newly installed, 0 to remove and 3 not upgraded.
Need to get 5,168 kB of archives.
After this operation, 25.6 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 python3-numpy amd64 1:1.21.5-1ubuntu22.04.1 [3,467 kB]
Get:2 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy/main amd64 python3-gdal amd64 3.8.4+dfsg-1~jammy0 [1,095 kB]
Get:3 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy/main amd64 gdal-bin amd64 3.8.4+dfsg-1~jammy0 [605 kB]
Fetched 5,168 kB in 2s (3,208 kB/s)
Selecting previously unselected package python3-numpy.
(Reading database ... 118243 file

## 6. Check CRS and Convert ASCII grids to GeoTIFF

Before clipping the rasters to the study area, we need to check the coordinate reference systems (CRS) of both datasets:

- the DWD raster grid
- the study-area shapefile

Both datasets must use the same CRS, or one of them must be reprojected before clipping. Otherwise, the raster and polygon may not overlap correctly. We will first take care of the .asc files.

The DWD files are provided as ASCII grids (`.asc`). These files often do not store CRS information directly. Therefore, we can also identify the CRS using the downloaded DWD metadata file. The shapefile CRS can be checked directly from its spatial metadata.

Ultimately, we want to convert the ASCII grids to GeoTIFF. Using GeoTIFF makes the files easier to process with common GIS and Python tools.


In [22]:
# Let's check the metadata for the first .asc file
!gdalinfo "$(find "{DWD_DATA_DIR}" -name "*.asc" | head -1)"

Driver: AAIGrid/Arc/Info ASCII Grid
Files: /content/opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/soil_moist/grids_germany_monthly_soil_moist_202508.asc
Size is 654, 866
Origin = (3280414.000000000000000,6103501.000000000000000)
Pixel Size = (1000.000000000000000,-1000.000000000000000)
Corner Coordinates:
Upper Left  ( 3280414.000, 6103501.000) 
Lower Left  ( 3280414.000, 5237501.000) 
Upper Right ( 3934414.000, 6103501.000) 
Lower Right ( 3934414.000, 5237501.000) 
Center      ( 3607414.000, 5670501.000) 
Band 1 Block=654x1 Type=Int32, ColorInterp=Undefined
  NoData Value=-9999


In [24]:
# We defined SOURCE_CRS as "EPSG:31467" in the settings at the beginning of the notebook.
# Here we use the shell environment variables created above.

!for f in $(find "$DWD_DATA_DIR" -name "*.asc"); do \
    gdal_translate -q -a_srs "$SOURCE_CRS" "$f" "${f%.asc}.tif"; \
done

# What the above command does:
# Find all decompressed DWD ASCII grid files and convert each one into a GeoTIFF
# while assigning the configured coordinate reference system.

In [25]:
# We can inspect the first 10 files in our folder again
os.listdir(DWD_DATA_DIR)[:10]

['grids_germany_monthly_soil_moist_202508.asc',
 'grids_germany_monthly_soil_moist_199807.tif',
 'grids_germany_monthly_soil_moist_201304.tif',
 'grids_germany_monthly_soil_moist_200706.asc',
 'grids_germany_monthly_soil_moist_202410.asc',
 'grids_germany_monthly_soil_moist_202210.asc',
 'BESCHREIBUNG_gridsgermany_monthly_soil_moist_de.pdf',
 'grids_germany_monthly_soil_moist_199809.asc',
 'grids_germany_monthly_soil_moist_200709.tif',
 'grids_germany_monthly_soil_moist_201107.tif']

In [26]:
# Let's have a closer look at what we got in our folder and files

# Check newly created GeoTIFF files
!find "$DWD_DATA_DIR" -name "*.tif" | head -5

/content/opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/soil_moist/grids_germany_monthly_soil_moist_199807.tif
/content/opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/soil_moist/grids_germany_monthly_soil_moist_201304.tif
/content/opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/soil_moist/grids_germany_monthly_soil_moist_200709.tif
/content/opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/soil_moist/grids_germany_monthly_soil_moist_201107.tif
/content/opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/soil_moist/grids_germany_monthly_soil_moist_200909.tif


In [27]:
# Count GeoTIFF files
!find "$DWD_DATA_DIR" -name "*.tif" | wc -l

426


In [28]:
# Count GeoTIFF files
!find "$DWD_DATA_DIR" -name "*.asc" | wc -l

426


In [29]:
# Inspect metadata and CRS of one converted GeoTIFF
!gdalinfo "$(find "$DWD_DATA_DIR" -name "*.tif" | head -1)" | head -80

Driver: GTiff/GeoTIFF
Files: /content/opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/soil_moist/grids_germany_monthly_soil_moist_199807.tif
Size is 654, 866
Coordinate System is:
PROJCRS["DHDN / 3-degree Gauss-Kruger zone 3",
    BASEGEOGCRS["DHDN",
        DATUM["Deutsches Hauptdreiecksnetz",
            ELLIPSOID["Bessel 1841",6377397.155,299.1528128,
                LENGTHUNIT["metre",1]]],
        PRIMEM["Greenwich",0,
            ANGLEUNIT["degree",0.0174532925199433]],
        ID["EPSG",4314]],
    CONVERSION["3-degree Gauss-Kruger zone 3",
        METHOD["Transverse Mercator",
            ID["EPSG",9807]],
        PARAMETER["Latitude of natural origin",0,
            ANGLEUNIT["degree",0.0174532925199433],
            ID["EPSG",8801]],
        PARAMETER["Longitude of natural origin",9,
            ANGLEUNIT["degree",0.0174532925199433],
            ID["EPSG",8802]],
        PARAMETER["Scale factor at natural origin",1,
            SCALEUNIT["unity",1],
           

In [32]:
# or get the CRS code using rasterio
# Expected CRS output should be something like:
# EPSG:31467

tif_files = glob.glob(os.path.join(os.environ["DWD_DATA_DIR"], "**", "*.tif"), recursive=True)

sample_tif = tif_files[0]

with rasterio.open(sample_tif) as src:
    print("Sample file:", sample_tif)
    print("CRS:", src.crs)

Sample file: /content/opendata.dwd.de/climate_environment/CDC/grids_germany/monthly/soil_moist/grids_germany_monthly_soil_moist_199807.tif
CRS: EPSG:31467


### Coordinate reference system

A coordinate reference system (CRS) defines how spatial coordinates are located on the Earth. Raster and vector data must use the same CRS, or be transformed to a common CRS, before spatial operations such as clipping.

The DWD ASCII grids do not always store CRS metadata directly in the `.asc` files. Therefore, the CRS is taken from the DWD metadata documentation. In this workflow, the DWD raster grid is assigned `EPSG:31467`, which corresponds to `DHDN / 3-degree Gauss-Krüger zone 3`.

`EPSG:31467` is not a CRS that every region automatically has. Instead, it is the CRS used by this DWD raster dataset. The study-area shapefile must either already use the same CRS or be reprojected to it before clipping.

In [33]:
# Lastly, let's clean up the files we do not need anylonger, meaning the .asc files

# Delete decompressed ASCII grids after successful GeoTIFF conversion
!find "$DWD_DATA_DIR" -name "*.asc" -delete

In [34]:
# Verify that ASC files were removed
!find "$DWD_DATA_DIR" -name "*.asc" | wc -l

0


## 7. Reproject the study-area shapefile and clip the rasters to the study area

Each GeoTIFF is clipped to the study-area shapefile.

Before clipping the raster files, the study-area shapefile must use the same CRS as the raster data.

The DWD GeoTIFFs were assigned `EPSG:31467` based on the DWD metadata. Therefore, the shapefile is reprojected to the same CRS before it is used as the clipping boundary. We will check if the shapefiles is already in a matching CRS or if we need to reproject the shapefile ourselves.

In [44]:
# Reproject shapefile to EPSG:31467 before clipping
# Read the study-area shapefile, we defined the path in the settings in the beginning of the notebook
study_area = gpd.read_file(SHAPEFILE)

print(f"Original shapefile CRS: {study_area.crs}")

Original shapefile CRS: EPSG:25832


In [45]:
# Reproject the shapefile to the same CRS as the DWD raster grid
study_area_31467 = study_area.to_crs(SOURCE_CRS)

REPROJECTED_SHAPEFILE = (
    SHAPEFILE.parent
    / f"{SHAPEFILE.stem}_{SOURCE_CRS.replace(':', '')}.shp"
)

study_area_31467.to_file(REPROJECTED_SHAPEFILE)

/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Field STAND created as String field, though DateTime requested.
  ogr_write(


In [48]:
# Check reprojected shapefile CRS
study_area_31467.crs

<Projected CRS: EPSG:31467>
Name: DHDN / 3-degree Gauss-Kruger zone 3
Axis Info [cartesian]:
- X[north]: Northing (metre)
- Y[east]: Easting (metre)
Area of Use:
- name: Germany - former West Germany onshore between 7°30'E and 10°30'E - states of Baden-Wurtemberg, Bayern, Bremen, Hamberg, Hessen, Niedersachsen, Nordrhein-Westfalen, Rhineland-Pfalz, Schleswig-Holstein.
- bounds: (7.5, 47.27, 10.51, 55.09)
Coordinate Operation:
- name: 3-degree Gauss-Kruger zone 3
- method: Transverse Mercator
Datum: Deutsches Hauptdreiecksnetz
- Ellipsoid: Bessel 1841
- Prime Meridian: Greenwich

In [53]:

CROPPED_DIR.mkdir(parents=True, exist_ok=True)

tif_files = list(DWD_DATA_DIR.rglob("*.tif"))

print(f"GeoTIFF files found: {len(tif_files)}")
print(f"Clipping shapefile: {REPROJECTED_SHAPEFILE}")

for tif_file in tif_files:
    output_name = tif_file.name.replace(
        "grids_germany_monthly_soil_moist_",
        "grids_cropped_monthly_soil_moist_",
    )

    output_file = CROPPED_DIR / output_name

    command = [
        "gdalwarp",
        "-q",
        "-cutline", str(REPROJECTED_SHAPEFILE),
        "-crop_to_cutline",
        "-dstalpha",
        str(tif_file),
        str(output_file),
    ]

    subprocess.run(command, check=True)

print(f"Clipped rasters saved to: {CROPPED_DIR}")

GeoTIFF files found: 426
Clipping shapefile: /content/shapefiles/kerpen_adm_25832/kerpen_adm_25832_EPSG31467.shp
Clipped rasters saved to: /content/cropped


## 8. Calculate the regional monthly mean

This step converts the clipped monthly rasters into a regional time series.

Each GeoTIFF represents one month, with the date encoded in the filename as `YYYYMM` such as `199101` for January 1991. The raster values describe soil moisture as percentage of plant-available water capacity (`% nFK`).

For each raster, the workflow reads the data, masks nodata values and pixels outside the study area, and calculates the mean of all remaining valid pixels. The result is one regional monthly mean value per file.

The resulting time series should be interpreted as a model-based soil moisture indicator for the selected region, not as direct field measurements. According to the DWD metadata, the model uses sandy loam with a wilting point of 13 vol.% and a field capacity of 37 vol.%.

The same averaging approach could also be applied to precipitation rasters. In that case, the result would represent regional average monthly precipitation depth, for example in mm/month, not total precipitation volume over the full area.

In [55]:
# Collect clipped GeoTIFF files
files = sorted(CROPPED_DIR.glob("*.tif"))

print(f"Clipped GeoTIFF files found: {len(files)}")

records = []

for file_path in files:
    with rasterio.open(file_path) as src:
        # Read the first raster band as a masked array.
        # This automatically masks nodata values defined in the raster metadata.
        data = src.read(1, masked=True)

        # gdalwarp created an alpha band during clipping.
        # Pixels outside the study-area polygon have alpha = 0 and should be ignored.
        if src.count > 1:
            alpha = src.read(src.count)
            data = np.ma.masked_where(alpha == 0, data)

        regional_mean = float(data.mean())

    # Extract YYYYMM date from the DWD filename
    # This assumes the filename contains the month as YYYYMM,
    # for example: grids_cropped_monthly_soil_moist_199101.tif
    # ...
    # If a different DWD dataset or filename pattern is used,
    # this regular expression may need to be adjusted.
    date_string = re.search(r"(\d{6})", file_path.name).group(1)
    date = pd.to_datetime(date_string, format="%Y%m")

    records.append({
        "date": date,
        "filename": file_path.name,
        VALUE_COLUMN: regional_mean,
        "unit": VALUE_UNIT,
    })

df = (
    pd.DataFrame(records)
    .sort_values("date")
    .reset_index(drop=True)
)

df.head()

Clipped GeoTIFF files found: 426


,date,filename,soil_moisture_nfk_percent,unit
0,1991-01-01,grids_cropped_monthly_soil_moist_199101.tif,105.000000,% nFK
1,1991-02-01,grids_cropped_monthly_soil_moist_199102.tif,100.864865,% nFK
2,1991-03-01,grids_cropped_monthly_soil_moist_199103.tif,94.144144,% nFK
3,1991-04-01,grids_cropped_monthly_soil_moist_199104.tif,78.432432,% nFK
4,1991-05-01,grids_cropped_monthly_soil_moist_199105.tif,67.738739,% nFK


## 9. Create the final time series table

The final table keeps only the date, value, and unit columns and sorts the records chronologically.

In [57]:
cleaned_df = (
    df[["date", VALUE_COLUMN, "unit"]]
    .copy()
    .sort_values(by="date")
    .reset_index(drop=True)
)

cleaned_df.head()

,date,soil_moisture_nfk_percent,unit
0,1991-01-01,105.000000,% nFK
1,1991-02-01,100.864865,% nFK
2,1991-03-01,94.144144,% nFK
3,1991-04-01,78.432432,% nFK
4,1991-05-01,67.738739,% nFK


In [58]:
cleaned_df.dtypes

,0
date,datetime64[ns]
soil_moisture_nfk_percent,float64
unit,object


## 10. Export the results as CSV time series

The cleaned time series is saved as a CSV file and can be used for further analysis or visualization.


In [59]:
cleaned_df.to_csv(OUTPUT_CSV, index=False)

print("Saved to:", OUTPUT_CSV)
cleaned_df.head()

Saved to: /content/soil_moisture_nfk_percent_Kerpen_1991_2025.csv


,date,soil_moisture_nfk_percent,unit
0,1991-01-01,105.000000,% nFK
1,1991-02-01,100.864865,% nFK
2,1991-03-01,94.144144,% nFK
3,1991-04-01,78.432432,% nFK
4,1991-05-01,67.738739,% nFK


## 11. Export GeoTiffs to NetCDF

The clipped monthly GeoTIFF files can be combined into a single NetCDF file.

Each GeoTIFF represents one month. The date is extracted from the filename using the `YYYYMM` pattern, for example `199101` for January 1991. The rasters are then stacked along a new `time` dimension.

The resulting NetCDF file stores the regional raster time series as a single multi-dimensional dataset with dimensions `time`, `y`, and `x`.

In [62]:
netcdf_layers = []

clipped_files = sorted(CROPPED_DIR.glob("*.tif"))

print(f"Clipped GeoTIFF files found: {len(clipped_files)}")

for file_path in clipped_files:
    # Extract the date from the filename.
    # This assumes the filename contains the month as YYYYMM,
    # for example: grids_cropped_monthly_soil_moist_199101.tif
    #
    # If a different DWD dataset or filename pattern is used,
    # this regular expression may need to be adjusted.
    date_match = re.search(r"(\d{6})", file_path.name)

    if date_match is None:
        raise ValueError(f"No YYYYMM date found in filename: {file_path.name}")

    date_string = date_match.group(1)
    date = pd.to_datetime(date_string, format="%Y%m")

    # Open the raster as an xarray DataArray.
    # masked=True converts nodata pixels to NaN.
    raster = rxr.open_rasterio(file_path, masked=True)

    # Select the first data band.
    # The clipped GeoTIFFs may also contain an alpha band from gdalwarp.
    raster = raster.sel(band=1).drop_vars("band")

    # Add the month as a time coordinate.
    raster = raster.expand_dims(time=[date])

    netcdf_layers.append(raster)

soil_moisture_cube = xr.concat(netcdf_layers, dim="time")

soil_moisture_cube.name = VALUE_COLUMN
soil_moisture_cube.attrs["units"] = VALUE_UNIT
soil_moisture_cube.attrs["long_name"] = "Monthly soil moisture as percentage of plant-available water capacity"
soil_moisture_cube.attrs["description"] = (
    "Regional clipped DWD monthly soil moisture rasters stacked along the time dimension."
)

soil_moisture_cube

Clipped GeoTIFF files found: 426


<xarray.DataArray 'soil_moisture_nfk_percent' (time: 426, y: 10, x: 17)> Size: 579kB
array([[[  0.,   0.,   0., ...,   0.,   0.,   0.],
        [  0.,   0.,   0., ...,   0.,   0.,   0.],
        [105., 105., 105., ...,   0.,   0.,   0.],
        ...,
        [105., 105., 105., ..., 105., 105., 105.],
        [  0.,   0.,   0., ..., 105., 105.,   0.],
        [  0.,   0.,   0., ...,   0.,   0.,   0.]],

       [[  0.,   0.,   0., ...,   0.,   0.,   0.],
        [  0.,   0.,   0., ...,   0.,   0.,   0.],
        [100., 100., 101., ...,   0.,   0.,   0.],
        ...,
        [101., 101., 101., ..., 101., 101., 101.],
        [  0.,   0.,   0., ..., 101., 101.,   0.],
        [  0.,   0.,   0., ...,   0.,   0.,   0.]],

       [[  0.,   0.,   0., ...,   0.,   0.,   0.],
        [  0.,   0.,   0., ...,   0.,   0.,   0.],
        [ 94.,  94.,  94., ...,   0.,   0.,   0.],
        ...,
...
        ...,
        [ 81.,  81.,  81., ...,  80.,  81.,  81.],
        [  0.,   0.,   0., ...,  80.,  80.,   0.],
        [  0.,   0.,   0., ...,   0.,   0.,   0.]],

       [[  0.,   0.,   0., ...,   0.,   0.,   0.],
        [  0.,   0.,   0., ...,   0.,   0.,   0.],
        [ 66.,  66.,  66., ...,   0.,   0.,   0.],
        ...,
        [ 66.,  66.,  66., ...,  66.,  67.,  67.],
        [  0.,   0.,   0., ...,  66.,  66.,   0.],
        [  0.,   0.,   0., ...,   0.,   0.,   0.]],

       [[  0.,   0.,   0., ...,   0.,   0.,   0.],
        [  0.,   0.,   0., ...,   0.,   0.,   0.],
        [ 54.,  54.,  54., ...,   0.,   0.,   0.],
        ...,
        [ 55.,  55.,  55., ...,  55.,  56.,  56.],
        [  0.,   0.,   0., ...,  55.,  55.,   0.],
        [  0.,   0.,   0., ...,   0.,   0.,   0.]]])
Coordinates:
  * time         (time) datetime64[ns] 3kB 1991-01-01 1991-02-01 ... 2026-06-01
  * y            (y) float64 80B 5.645e+06 5.644e+06 ... 5.637e+06 5.636e+06
  * x            (x) float64 136B 3.329e+06 3.33e+06 ... 3.344e+06 3.345e+06
    spatial_ref  int64 8B 0
Attributes:
    AREA_OR_POINT:  Area
    scale_factor:   1.0
    add_offset:     0.0
    units:          % nFK
    long_name:      Monthly soil moisture as percentage of plant-available wa...
    description:    Regional clipped DWD monthly soil moisture rasters stacke...

In [63]:
OUTPUT_NETCDF = WORK_DIR / f"{VALUE_COLUMN}_{REGION_NAME}_{START_YEAR}_{END_YEAR}.nc"

soil_moisture_cube.to_netcdf(OUTPUT_NETCDF)

print(f"NetCDF saved to: {OUTPUT_NETCDF}")

NetCDF saved to: /content/soil_moisture_nfk_percent_Kerpen_1991_2025.nc


/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: SerializationWarning: saving variable soil_moisture_nfk_percent with floating point data as an integer dtype without any _FillValue to use for NaNs
  exec(code_obj, self.user_global_ns, self.user_ns)


In [65]:
# Let's have a look at the file we just created
ds_check = xr.open_dataset(OUTPUT_NETCDF)
ds_check

<xarray.Dataset> Size: 583kB
Dimensions:                    (time: 426, y: 10, x: 17)
Coordinates:
  * time                       (time) datetime64[ns] 3kB 1991-01-01 ... 2026-...
  * y                          (y) float64 80B 5.645e+06 5.644e+06 ... 5.636e+06
  * x                          (x) float64 136B 3.329e+06 3.33e+06 ... 3.345e+06
Data variables:
    spatial_ref                int64 8B ...
    soil_moisture_nfk_percent  (time, y, x) float64 579kB ...

In [66]:
print("Start:", ds_check.time.min().values)
print("End:", ds_check.time.max().values)
print("Time steps:", ds_check.sizes["time"])

Start: 1991-01-01T00:00:00.000000000
End: 2026-06-01T00:00:00.000000000
Time steps: 426
